# Notebook 06 — Wiring the Wholesale UI

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS — Workshop 2

---

How does a React app consume registered agents and a FIBO-shaped API together?

This notebook demonstrates the two-driver architecture of an agentic AI
application: GraphQL drives what data is rendered, and the Agent Registry drives
what actions are available. You will see the permission model become visible in
the rendered UI — and be precise about which layers are *enforced today* (access
control) versus *roadmap* (per-row data scoping).

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
# Skips uv sync (which installs the full agent stack and takes minutes).
import sys, subprocess

# Only install packages not already provided by the SageMaker base image
pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0']

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Two-driver architecture** | The pattern where a UI is driven by two independent sources: (1) a GraphQL API that provides *data* and (2) the Agent Registry that provides *capabilities*. Neither is hardcoded; both are queried live. |
| **Capability palette** | The panel in the UI that shows what actions the current user can take. Populated from the Agent Registry filtered by persona claim. Different personas see different palettes. This is a registry *display* of what the persona may invoke; click-to-invoke from the palette is roadmap. |
| **Entity 360** | A full-page view of a single entity (Customer, Household) showing all related data: accounts, signals, relationships, audit trail. The primary screen in the Wholesale UI. |
| **Provenance card** | A UI component that shows where a piece of data came from: which SHACL shape validated it, which R2RML mapping produced the underlying triple, which agent generated it. Provenance is visible, not hidden. |
| **Compliance banner** | The banner at the top of the Referral Detail screen. Reads *"Active compliance review — contact BSA team before client outreach."* Never reads *"SAR filed"* (31 U.S.C. §5318(g)(2) tipping-off prohibition). |
| **Permission model — enforced vs. roadmap** | The *enforced* layers today are: Identity (IDC) → Application (Cognito persona claim) → **field/capability access** (AppSync authorizes by Cognito group; agents carry `VALID_PERSONAS` allow-lists) → Semantic (SHACL on writes/decisions). The persona claim is **validated** for access control — it gates *which fields/capabilities a caller may invoke*. Per-row **Lake Formation** scoping (the *same* query returning *different rows* per persona) is **roadmap, not enforced**: the direct-Neptune read path returns the same rows regardless of persona. Do not claim row-level scoping is live. |

## Two drivers, one screen

Most enterprise applications have one driver: an API that returns data, and the
UI renders it. The set of actions available to the user is hardcoded in the UI
code — a button exists because a developer put it there, not because the system
decided it should be there.

The Wholesale UI has two drivers. The first is the FIBO-shaped GraphQL API from
notebook 04 — it provides the data that populates the Entity 360 screen. The
second is the Agent Registry from notebook 03 — it provides the capability
palette that tells the UI which actions to render for the current user.

This separation matters because it makes the UI *evolvable without redeploy*.
When a new agent is registered (say, a Phase 2 behavioral-signal-agent), the
Wholesale UI's capability palette updates automatically — the registry returns
the new agent in its discovery response, and the UI renders a new button. No
code change. No deploy. The registry is the source of truth for what actions
exist.

### The permission model becomes visible — be precise about it

The persona claim shapes what the user sees. Be exact about *how*, because it is
tempting to overclaim. When **Dana Brooks (the Consumer Banker)** signs in, she
sees:
- Her book of clients — the customer cards the dashboard renders (Data layer)
- The Consumer Banking routes (Application layer: Cognito groups)
- The capability palette with "Route to advisor" and "Draft rationale" (Registry: persona-scoped — the actions her persona may invoke)
- Wealth signals with provenance (Semantic layer: SHACL-validated, named graphs)

When a **BSA Analyst** signs in to the same URL, they see a different capability
palette — no "Route to advisor," but compliance-oriented actions instead — and
the compliance fields the Consumer Banker is not authorized to call. The UI is
one codebase; the persona claim composes a different view.

**What is enforced today, stated honestly:** the layers that actually gate the
view are Identity (IDC) → Application (Cognito persona claim) → **field/capability
access** (AppSync authorizes every request by Cognito group; the action agents
carry `VALID_PERSONAS` allow-lists; the persona is *validated*) → Semantic (SHACL
on writes and decisions). The persona claim decides *which fields and capabilities
a caller may invoke* — a BSA Analyst can call compliance fields a Consumer Banker
cannot.

**What is roadmap, not enforced:** per-row **Lake Formation** scoping — the *same*
query returning a *different set of rows* depending on persona — is **not** wired
in the running system. The direct-Neptune read path does not pass the persona
claim down to a row filter; it returns the same rows regardless of caller. The
design target is Lake Formation-scoped Iceberg via Ontop, but that is future work.
**Do not claim row-level scoping is live.** So: persona decides *who can call what*
(real, today); persona deciding *which rows you see* is the roadmap. This is the
honest version of the four-layer model — the access-control layer is enforced; the
data-scoping layer is designed but not yet enforced.

One critical regulatory constraint governs the UI: the compliance banner. When
a household is under active compliance review, the banner reads *"Active
compliance review — contact BSA team before client outreach."* It never reads
*"SAR filed"* because 31 U.S.C. §5318(g)(2) makes it a federal crime to
disclose to anyone outside the BSA function that a SAR has been filed. The
Consumer Banker is outside the BSA function. The banner tells them what they
are allowed to know — that a review is active — without disclosing what kind
of review it is.

In [ ]:
import sys
import os
import json

# Workshop 1's shared helpers
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

# Load agent descriptors for capability palette simulation
SPEC_DIR = "../../spec/04-aws-agent-registry"

def load_descriptors(subdir):
    path = os.path.join(SPEC_DIR, subdir)
    descriptors = []
    for f in sorted(os.listdir(path)):
        if f.endswith(".json"):
            with open(os.path.join(path, f)) as fh:
                descriptors.append(json.load(fh))
    return descriptors

agent_descriptors = load_descriptors("agents")
print(f"Loaded {len(agent_descriptors)} agent descriptors")

In [ ]:
# Build cell 1 — Simulate the Entity 360 data fetch.
#
# In the real UI, this is a GraphQL query. Here we simulate the
# response to show what the UI renders for the Patel household.

entity_360_data = {
    "household": {
        "uri": "atlas:hh/9c2a1e",
        "label": "Patel Household",
        "members": [
            {"uri": "atlas:cust/9c2a1e", "label": "Anjali Patel", "customerId": "CUST-9C2A1E"},
            {"uri": "atlas:cust/7b3f2d", "label": "Raj Patel", "customerId": "CUST-7B3F2D"},
        ],
    },
    "signals": [
        {"signalType": "LargeInboundWireSignal", "strength": "strong",
         "provenance": {"validatedBy": "atlas:WealthSignalTypeShape", "derivedFrom": "pattern_a_iceberg/customer-master.r2rml.ttl"}},
        {"signalType": "NoAdvisorCoverageSignal", "strength": "gap",
         "provenance": {"validatedBy": "atlas:WealthSignalTypeShape", "derivedFrom": "pattern_a_iceberg/advisory-relationships.r2rml.ttl"}},
    ],
    "advisoryRelationships": [],
    "complianceReview": True,
}

print("Entity 360 — Patel Household")
print("=" * 50)
print(f"Household: {entity_360_data['household']['label']}")
print(f"Members:   {', '.join(m['label'] for m in entity_360_data['household']['members'])}")
print(f"\nWealth Signals ({len(entity_360_data['signals'])}):\n")
for sig in entity_360_data["signals"]:
    print(f"  • {sig['signalType']} (strength: {sig['strength']})")
    print(f"    Validated by: {sig['provenance']['validatedBy']}")
    print(f"    Derived from: {sig['provenance']['derivedFrom']}")
print(f"\nAdvisor Coverage: {'None (gap)' if not entity_360_data['advisoryRelationships'] else 'Active'}")

In [ ]:
# Build cell 2 — The compliance banner.
#
# 31 U.S.C. §5318(g)(2) makes it a federal crime to disclose that a
# SAR has been filed to anyone outside the BSA function. The Consumer
# Banker is outside the BSA function. The banner tells them what they
# are allowed to know.

def render_compliance_banner(has_compliance_review, persona_claim):
    """Render the compliance banner based on review status and persona."""
    if not has_compliance_review:
        return None  # No banner needed
    
    if persona_claim == "atlas-bsa-analyst":
        # BSA Analyst can see the full detail
        return "⚠️  SAR draft in progress — BSA team review required before filing"
    else:
        # Everyone else sees the safe version
        # NEVER say "SAR filed" — that's tipping off (federal crime)
        return "⚠️  Active compliance review — contact BSA team before client outreach"

# Consumer Banker view
banner_banker = render_compliance_banner(True, "atlas-consumer-banker")
print(f"Consumer Banker sees: {banner_banker}")
print()

# BSA Analyst view
banner_bsa = render_compliance_banner(True, "atlas-bsa-analyst")
print(f"BSA Analyst sees:     {banner_bsa}")
print()
print("The difference is 31 U.S.C. §5318(g)(2): the tipping-off prohibition.")
print("The Consumer Banker cannot know whether the review involves a SAR.")

In [ ]:
# Build cell 3 — The capability palette (from Agent Registry).
#
# This is the second driver: the registry tells the UI what actions
# to render. Different personas see different palettes.

def get_capability_palette(persona_claim):
    """Query the registry for capabilities discoverable by this persona."""
    phase_1_agents = [d for d in agent_descriptors if d.get("phase") == 1]
    capabilities = []
    for desc in phase_1_agents:
        discoverable_by = desc.get("registry_metadata", {}).get("discoverable_by", [])
        if persona_claim in discoverable_by:
            capabilities.append({
                "name": desc["agent_name"],
                "displayName": desc["registry_metadata"]["display_name"],
                "displayIcon": desc["registry_metadata"]["display_icon"],
                "posture": desc["posture"],
                "capabilityTag": desc["registry_metadata"]["capability_tag"],
            })
    return capabilities

# Consumer Banker palette
banker_palette = get_capability_palette("atlas-consumer-banker")
print("Consumer Banker capability palette:")
print("-" * 50)
for cap in banker_palette:
    print(f"  [{cap['displayIcon']}] {cap['displayName']}")
    print(f"      Agent: {cap['name']} | Posture: {cap['posture']}")
    print()

# Wealth Advisor palette (for comparison)
advisor_palette = get_capability_palette("atlas-wealth-advisor")
print("\nWealth Advisor capability palette:")
print("-" * 50)
for cap in advisor_palette:
    print(f"  [{cap['displayIcon']}] {cap['displayName']}")
    print(f"      Agent: {cap['name']} | Posture: {cap['posture']}")
    print()

In [ ]:
# Build cell 4 — Simulating the "Route to advisor" action.
#
# When the Consumer Banker clicks "Route to advisor" in the capability
# palette, the UI invokes the referral-orchestrator agent through the
# registry. The agent requires an approved rationale (human-in-the-loop).

def simulate_route_to_advisor(household_uri, signals, approved_rationale, banker_id):
    """Simulate what happens when the banker clicks 'Route to advisor'."""
    print("  Step 1: UI invokes referral-orchestrator via Agent Registry")
    print(f"  Step 2: Orchestrator receives approved rationale ({len(approved_rationale)} chars)")
    print(f"  Step 3: select-advisor queries SLGD for eligible advisors")
    print(f"  Step 4: validate-routing checks SHACL RoutingPolicyShape")
    print(f"  Step 5: write-routing-decision writes atlas:RoutingDecision to SLGD")
    print(f"  Step 6: notify-advisor emits CloudWatch event")
    print(f"  Step 7: audit-write records atlas:AuditRecord with PROV-O")
    return {
        "status": "routed",
        "selected_advisor": "Marcus Webb (Wealth Advisory)",
        "routing_decision_uri": "atlas:routing/demo-001",
        "audit_record_uri": "atlas:audit/demo-001",
    }

# Dana Brooks (the Consumer Banker) routes the customer Rachel Kim's household.
print("Simulating 'Route to advisor' workflow:\n")
result = simulate_route_to_advisor(
    household_uri="atlas:hh/9c2a1e",
    signals=["atlas:signal/wire-001", "atlas:signal/gap-001"],
    approved_rationale="The Patel household shows strong wealth-readiness signals...",
    banker_id="dana.brooks",  # the Consumer Banker who signs in (Rachel Kim is the CUSTOMER)
)
print(f"\n  Result: {result['status']}")
print(f"  Advisor: {result['selected_advisor']}")
print(f"  Audit:   {result['audit_record_uri']}")

## The demo loop — Dana's half (route → draft → approve → it lands as "new")

Dana's actions above are the first half of the workshop's end-to-end demo loop. From the
Wholesale UI, as **Dana Brooks (the Consumer Banker)**:

1. **Open a signalled customer.** She opens the customer **Rachel Kim** — flagged with two
   *derived* signals (a *Large Deposit Pattern* and *No Advisor Coverage*; these are ATLAS
   outputs computed from the graph, not inputs — see
   [`05_wealth_signals.ipynb`](./05_wealth_signals.ipynb)) and "No advisory coverage."
2. **Route referral → Generate draft.** The `referral-rationale-drafter` writes a rationale
   grounded in Rachel's *actual* signals. It tolerates a household with no signals yet
   (drafting from the real household members instead) and returns an honest
   *insufficient-context* rather than inventing one — it is badged **probabilistic, requires
   human review**.
3. **Approve and route.** Dana's approval is the human-in-the-loop gate (the
   `referral-orchestrator` requires `approved_rationale`, verified below). Approving starts
   the workflow, which writes a SHACL-validated `RoutingDecision` and assigns Marcus Webb as
   Rachel's advisor.

**The outcome Dana triggers:** Rachel arrives in **Marcus Webb (the Wealth Advisor)**'s book
flagged **"New — routed to you."** That banner is a *real* state, not a timer: it shows while
the relationship was created by the routing workflow (`routedByWorkflow`) and Marcus has not
yet accepted it (`takenOnAt` is null). Marcus's half — seeing the banner, **Take on client**
(which writes a real `atlas:takenOnAt` and clears the banner), and the workshop **Reset** —
is taught from the advisor's side in
[`../phase-2-advisor/03_wealth_ui.ipynb`](../phase-2-advisor/03_wealth_ui.ipynb), and the
**full cross-persona walk** (route → banner → take-on → clear → reset) is in
[`../phase-2-advisor/05_end_to_end.ipynb`](../phase-2-advisor/05_end_to_end.ipynb) and the
canonical demo script [`DEMO.md`](../../DEMO.md) (with the presenter runbook
[`../phase-2-advisor/07_demo_runbook.ipynb`](../phase-2-advisor/07_demo_runbook.ipynb)).

Switching between the two personas in the live demo is just **Sign out** (top-right) and
signing in as the other user.

## Verification

Three things must be true for the Wholesale UI to be correct:

1. The capability palette is persona-scoped (Consumer Banker sees different actions than Wealth Advisor)
2. The compliance banner respects the tipping-off prohibition
3. The "Route to advisor" action requires an approved rationale (human-in-the-loop)

In [ ]:
# Verification cell 1 — Capability palette is persona-scoped.
#
# Consumer Banker must see "Route to advisor" and "Draft referral rationale".
# Wealth Advisor must NOT see "Draft referral rationale".

banker_names = {c["name"] for c in banker_palette}
advisor_names = {c["name"] for c in advisor_palette}

assert "referral-orchestrator" in banker_names, \
    "Consumer Banker must see referral-orchestrator"
assert "referral-rationale-drafter" in banker_names, \
    "Consumer Banker must see referral-rationale-drafter"
assert "referral-rationale-drafter" not in advisor_names, \
    "Wealth Advisor must NOT see referral-rationale-drafter"

print("✓ Capability palette is correctly persona-scoped.")
print(f"  Consumer Banker: {sorted(banker_names)}")
print(f"  Wealth Advisor:  {sorted(advisor_names)}")

In [ ]:
# Verification cell 2 — Compliance banner respects tipping-off prohibition.
#
# The banner for non-BSA personas must NEVER contain "SAR" or "filed".
# If this fails: you have a tipping-off violation (federal crime).

non_bsa_personas = ["atlas-consumer-banker", "atlas-wealth-advisor", "atlas-ontology-steward"]

for persona in non_bsa_personas:
    banner = render_compliance_banner(True, persona)
    assert "SAR" not in banner, \
        f"TIPPING-OFF VIOLATION: Banner for {persona} contains 'SAR'"
    assert "filed" not in banner.lower(), \
        f"TIPPING-OFF VIOLATION: Banner for {persona} contains 'filed'"

# BSA Analyst CAN see SAR details
bsa_banner = render_compliance_banner(True, "atlas-bsa-analyst")
assert "SAR" in bsa_banner, "BSA Analyst should see SAR details"

print("✓ Compliance banner respects 31 U.S.C. §5318(g)(2) tipping-off prohibition.")
print("  Non-BSA personas see: 'Active compliance review'")
print("  BSA Analyst sees:     SAR-specific detail")

In [ ]:
# Verification cell 3 — Route to advisor requires approved rationale.
#
# The referral-orchestrator's input_schema requires 'approved_rationale'.
# This enforces the human-in-the-loop pattern: no auto-routing.

orchestrator_desc = next(
    d for d in agent_descriptors if d["agent_name"] == "referral-orchestrator"
)
required_fields = orchestrator_desc["input_schema"]["required"]

assert "approved_rationale" in required_fields, \
    "referral-orchestrator must require approved_rationale (human-in-the-loop)"
assert "persona_claim" in required_fields, \
    "referral-orchestrator must require persona_claim"

# Verify persona is restricted to consumer-banker
persona_schema = orchestrator_desc["input_schema"]["properties"]["persona_claim"]
assert persona_schema.get("const") == "atlas-consumer-banker", \
    "Only atlas-consumer-banker can invoke referral-orchestrator"

print("✓ Human-in-the-loop enforced: referral-orchestrator requires approved_rationale.")
print("  No path exists to auto-route without human approval.")
print(f"  Required fields: {required_fields}")

## What just changed

You have seen the two-driver architecture in action: GraphQL provides data,
the Agent Registry provides capabilities, and the persona claim composes
different views for different personas from the same codebase. You also saw
the honest shape of the permission model: access control — *which fields and
capabilities a persona may invoke* — is enforced today (Cognito groups +
agent `VALID_PERSONAS` allow-lists + SHACL on writes); per-row Lake Formation
data scoping is roadmap, not enforced (the read path returns the same rows
regardless of persona).

You have also seen the regulatory constraint that governs the UI: the tipping-off
prohibition means the compliance banner can never disclose SAR status to non-BSA
personas. This is not a UX choice — it is a legal requirement.

Phase 1 is now functionally complete. The next notebook runs the full acceptance
suite to confirm every contract holds before you move to Phase 2.